# Eval Workbench

This notebook runs the default eval workflow, reloads the generated artifacts from `outputs/`, and renders summary tables for quick iteration after code changes.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / "app").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if not (repo_root / "app").exists():
    raise RuntimeError("Could not locate the repository root from the current working directory.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

from app.notebook_eval import (
    build_item_error_table,
    build_persona_error_table,
    load_eval_metrics,
    load_eval_records,
    run_eval_notebook,
    split_item_error_table,
)

repo_root


PosixPath('/home/mdel2424/dev/eRisk_Honours')

In [2]:
# Matches the default README eval command.
personas = 10
seed = 42
eval_mode = "mixed_holdout"
prompt_version = "v1"
max_api_calls = 500
fit_calibrator_policy = "auto"
trace_level = "off"
save_diagnostics = False
debug_outputs = False
randomize_eval_split = True
run_eval_now = True

output_dir = repo_root / "outputs"


In [3]:
if run_eval_now:
    run_summary = run_eval_notebook(
        persona_count=personas,
        seed=seed,
        eval_mode=eval_mode,
        prompt_version=prompt_version,
        save_diagnostics=save_diagnostics,
        max_api_calls=max_api_calls,
        trace_level=trace_level,
        fit_calibrator_policy=fit_calibrator_policy,
        randomize_eval_split=randomize_eval_split,
        debug_outputs=debug_outputs,
        output_dir=output_dir,
    )
    resolved_output_dir = Path(run_summary["output_dir"])
else:
    run_summary = None
    resolved_output_dir = Path(output_dir)

print(f"Artifacts ready in: {resolved_output_dir}")
run_summary if run_summary is not None else {"output_dir": str(resolved_output_dir)}


Running eval: mode=mixed_holdout, personas=10, prompt=v1, live_status=on
Eval split: mode=random_per_run | split_seed=717477627
[eval 1/4 persona=1] cycle=41 turn=39 stage=detector_graph conf=24.3% calls=75/500
[eval 2/4 persona=7] cycle=41 turn=39 stage=detector_graph conf=4.7% calls=152/500
[eval 3/4 persona=4] cycle=28 turn=26 stage=detector_graph conf=37.2% calls=202/500
[eval 4/4 persona=9] cycle=38 turn=36 stage=detector_graph conf=39.1% calls=268/500
item_f1=0.6202 objective=0.4852
Artifacts ready in: /home/mdel2424/dev/eRisk_Honours/outputs


{'metrics': {'eval_mode_requested': 'mixed_holdout',
  'eval_mode_effective': 'synthetic_holdout',
  'prompt_version': 'v1',
  'synthetic_train_count': 6,
  'synthetic_val_count': 2,
  'synthetic_test_count': 2,
  'split_counts': {'train': 6, 'val': 2, 'test': 2},
  'synthetic_val': {'binary_accuracy': None,
   'binary_f1': None,
   'bdi_mae': 3.5,
   'symptom_f1_at_4': 0.463,
   'item_f1_macro_at_1': 0.463,
   'item_mae': 0.5476,
   'headline_f1': 0.463,
   'avg_turns_to_decision': 40.0,
   'risk_recall': None,
   'binary_f1_defined': False,
   'risk_recall_defined': False,
   'metric_mode': 'item_only',
   'objective': 0.313},
  'synthetic_test': {'binary_accuracy': None,
   'binary_f1': None,
   'bdi_mae': 4.5,
   'symptom_f1_at_4': 0.6508,
   'item_f1_macro_at_1': 0.6508,
   'item_mae': 1.119,
   'headline_f1': 0.6508,
   'avg_turns_to_decision': 32.0,
   'risk_recall': None,
   'binary_f1_defined': False,
   'risk_recall_defined': False,
   'metric_mode': 'item_only',
   'objectiv

In [4]:
if "resolved_output_dir" not in globals():
    resolved_output_dir = Path(output_dir)

metrics_payload = load_eval_metrics(resolved_output_dir)
records_df = load_eval_records(resolved_output_dir)
persona_error_df = build_persona_error_table(records_df)
item_error_df = build_item_error_table(records_df)
item_error_views = split_item_error_table(item_error_df)

print(f"Loaded {len(records_df)} evaluated personas from {resolved_output_dir}")
records_df[["persona_id", "split", "family", "bdi_true", "bdi_pred"]].head()


Loaded 4 evaluated personas from /home/mdel2424/dev/eRisk_Honours/outputs


,persona_id,split,family,bdi_true,bdi_pred
0,1,val,risk_leaning,26,22
1,7,val,control_neutral,1,4
2,4,test,cognitive_ruminative,24,29
3,9,test,mixed_moderate,31,27


In [5]:
primary_metrics = dict(metrics_payload.get("primary_metrics", {}))
summary_metrics_df = pd.DataFrame(
    [
        {"metric": "primary_eval_split", "value": metrics_payload.get("primary_eval_split", "")},
        {"metric": "item_f1_macro_at_1", "value": primary_metrics.get("item_f1_macro_at_1", metrics_payload.get("item_f1_macro_at_1", 0.0))},
        {"metric": "item_mae", "value": primary_metrics.get("item_mae", metrics_payload.get("item_mae", 0.0))},
        {"metric": "bdi_mae", "value": primary_metrics.get("bdi_mae", metrics_payload.get("bdi_mae", 0.0))},
        {"metric": "avg_turns_to_decision", "value": primary_metrics.get("avg_turns_to_decision", metrics_payload.get("avg_turns_to_decision", 0.0))},
        {"metric": "objective", "value": primary_metrics.get("objective", metrics_payload.get("objective", 0.0))},
    ]
)
summary_metrics_df


,metric,value
0,primary_eval_split,overall_labeled
1,item_f1_macro_at_1,0.6202
2,item_mae,0.8333
3,bdi_mae,4.0
4,avg_turns_to_decision,36.0
5,objective,0.4852


In [6]:
family_summary_df = (
    persona_error_df.groupby("family", dropna=False)
    .agg(
        profiles=("persona_id", "count"),
        avg_bdi_true=("bdi_true", "mean"),
        avg_bdi_pred=("bdi_pred", "mean"),
        avg_bdi_abs_error=("bdi_abs_error", "mean"),
    )
    .reset_index()
    .sort_values(["avg_bdi_abs_error", "family"], ascending=[False, True])
)
family_summary_df.style.format(
    {
        "avg_bdi_true": "{:.2f}",
        "avg_bdi_pred": "{:.2f}",
        "avg_bdi_abs_error": "{:.2f}",
    }
)


,family,profiles,avg_bdi_true,avg_bdi_pred,avg_bdi_abs_error
0,cognitive_ruminative,1,24.00,29.00,5.00
2,mixed_moderate,1,31.00,27.00,4.00
3,risk_leaning,1,26.00,22.00,4.00
1,control_neutral,1,1.00,4.00,3.00


In [7]:
persona_error_df.head(15).style.format(
    {
        "bdi_true": "{:.0f}",
        "bdi_pred": "{:.0f}",
        "bdi_error": "{:+.0f}",
        "bdi_abs_error": "{:.0f}",
    }
)


,persona_id,split,family,source,bdi_true,bdi_pred,bdi_error,bdi_abs_error
0,4,test,cognitive_ruminative,synthetic,24,29,+5,5
1,1,val,risk_leaning,synthetic,26,22,-4,4
2,9,test,mixed_moderate,synthetic,31,27,-4,4
3,7,val,control_neutral,synthetic,1,4,+3,3


## Item-Level Analysis

`mean_error = avg_predicted_item_score - avg_ground_truth_item_score`

- Negative values mean under-predicted items.
- Positive values mean over-predicted items.


In [8]:
DISPLAY_COLUMNS = [
    "item_id",
    "symptom_name",
    "avg_pred",
    "avg_true",
    "mean_error",
    "abs_mean_error",
    "n_profiles",
]

def _hex_to_rgb(value: str):
    value = value.lstrip("#")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))

def _rgb_to_hex(rgb):
    return "#%02x%02x%02x" % tuple(max(0, min(255, int(channel))) for channel in rgb)

def _blend_colors(start_hex: str, end_hex: str, weight: float):
    start_rgb = _hex_to_rgb(start_hex)
    end_rgb = _hex_to_rgb(end_hex)
    weight = max(0.0, min(1.0, float(weight)))
    blended = [start + ((end - start) * weight) for start, end in zip(start_rgb, end_rgb)]
    return _rgb_to_hex(blended)

def _mean_error_style(value: float, scale: float) -> str:
    if pd.isna(value):
        return ""
    magnitude = min(abs(float(value)) / max(scale, 0.001), 1.0)
    if float(value) < 0:
        color = _blend_colors("#ffffff", "#d73027", magnitude)
    elif float(value) > 0:
        color = _blend_colors("#ffffff", "#4575b4", magnitude)
    else:
        color = "#ffffff"
    return f"background-color: {color};"

def style_item_error_table(frame: pd.DataFrame):
    if frame.empty:
        return frame.reindex(columns=DISPLAY_COLUMNS)
    scale = max(abs(float(frame["mean_error"].min())), abs(float(frame["mean_error"].max())), 0.001)
    return (
        frame[DISPLAY_COLUMNS]
        .style
        .format(
            {
                "avg_pred": "{:.3f}",
                "avg_true": "{:.3f}",
                "mean_error": "{:+.3f}",
                "abs_mean_error": "{:.3f}",
            }
        )
        .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])
    )


In [9]:
style_item_error_table(item_error_views["all_items"])


/tmp/ipykernel_62206/1530208316.py:52: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,19,Concentration Difficulty,0.500,1.750,-1.250,1.250,4
1,3,Past Failure,1.000,2.000,-1.000,1.000,4
2,5,Guilty Feelings,0.750,1.750,-1.000,1.000,4
3,4,Loss of Pleasure,1.250,2.000,-0.750,0.750,4
4,14,Worthlessness,0.750,1.500,-0.750,0.750,4
5,8,Self-Criticalness,1.250,1.750,-0.500,0.500,4
6,15,Loss of Energy,1.500,2.000,-0.500,0.500,4
7,2,Pessimism,1.250,1.500,-0.250,0.250,4
8,12,Loss of Interest,1.000,1.250,-0.250,0.250,4
9,16,Changes in Sleeping Pattern,1.000,1.250,-0.250,0.250,4


In [10]:
style_item_error_table(item_error_views["under_predicted"])


/tmp/ipykernel_62206/1530208316.py:52: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,19,Concentration Difficulty,0.500,1.750,-1.250,1.250,4
1,3,Past Failure,1.000,2.000,-1.000,1.000,4
2,5,Guilty Feelings,0.750,1.750,-1.000,1.000,4
3,4,Loss of Pleasure,1.250,2.000,-0.750,0.750,4
4,14,Worthlessness,0.750,1.500,-0.750,0.750,4
5,8,Self-Criticalness,1.250,1.750,-0.500,0.500,4
6,15,Loss of Energy,1.500,2.000,-0.500,0.500,4
7,2,Pessimism,1.250,1.500,-0.250,0.250,4
8,12,Loss of Interest,1.000,1.250,-0.250,0.250,4
9,16,Changes in Sleeping Pattern,1.000,1.250,-0.250,0.250,4


In [11]:
style_item_error_table(item_error_views["over_predicted"])


/tmp/ipykernel_62206/1530208316.py:52: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])


,item_id,symptom_name,avg_pred,avg_true,mean_error,abs_mean_error,n_profiles
0,20,Tiredness or Fatigue,1.500,0.250,+1.250,1.250,4
1,6,Punishment Feelings,1.000,0.000,+1.000,1.000,4
2,10,Crying,1.000,0.000,+1.000,1.000,4
3,11,Agitation,1.250,0.250,+1.000,1.000,4
4,9,Suicidal Thoughts or Wishes,1.500,0.750,+0.750,0.750,4
5,17,Irritability,0.750,0.000,+0.750,0.750,4
6,1,Sadness,0.750,0.250,+0.500,0.500,4
7,7,Self-Dislike,1.000,0.750,+0.250,0.250,4
8,13,Indecisiveness,0.500,0.250,+0.250,0.250,4
